In [1]:
from pathlib import Path
from datetime import datetime
import json
import platform

import joblib
import numpy as np
import pandas as pd
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupShuffleSplit,
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "Semana_1_Proyecto_3":
    PROJECT_DIR = PROJECT_DIR / "Semana_1_Proyecto_3"

DATA_PATH = PROJECT_DIR / "Data_ansiedad.xlsx"
MODEL_PATH = PROJECT_DIR / "rl_model_ansiedad.joblib"
METADATA_PATH = PROJECT_DIR / "rl_model_ansiedad_metadata.json"

FEATURES = ["P_A1", "P_A2", "P_A3", "P_A4", "P_A5", "P_A6"]
TARGET = "NIVEL_ANSIEDAD"
RANDOM_STATE = 42

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontro el archivo: {DATA_PATH}")

data = pd.read_excel(DATA_PATH)
print(f"Dataset cargado: {DATA_PATH.name}")
print(f"Filas: {data.shape[0]} | Columnas: {data.shape[1]}")
data.head()

Dataset cargado: Data_ansiedad.xlsx
Filas: 204 | Columnas: 12


,Fecha,correo,Sexo,Edad,P_A1,P_A2,P_A3,P_A4,P_A5,P_A6,SUMA,NIVEL_ANSIEDAD
0,2023-04-24 19:20:09.426,cdiegoo@ucvvirtual.edu.pe,Femenino,32,1,2,1,2,1,2,9,ANSIEDAD MODERADA
1,2023-04-24 19:24:57.114,cdiegoo@ucvvirtual.edu.pe,Femenino,30,2,2,1,2,1,2,10,ANSIEDAD MODERADA
2,2023-04-25 15:56:39.289,allfrancisco2191@gmail.com,Masculino,31,3,3,3,3,3,3,18,ANSIEDAD GRAVE
3,2023-04-25 15:59:06.943,carlos.javier.canales@gmail.com,Masculino,38,0,0,0,0,0,0,0,SIN ANSIEDAD
4,2023-04-25 16:18:50.978,gabriel_your_daddy@hotmail.com,Masculino,42,1,2,1,1,1,0,6,ANSIEDAD LEVE


In [4]:
missing_values = data[FEATURES + [TARGET]].isnull().sum()
print("Valores faltantes por columna:")
print(missing_values)

print("\nDistribucion de clases:")
print(data[TARGET].value_counts(dropna=False))

Valores faltantes por columna:
P_A1              0
P_A2              0
P_A3              0
P_A4              0
P_A5              0
P_A6              0
NIVEL_ANSIEDAD    0
dtype: int64

Distribucion de clases:
NIVEL_ANSIEDAD
SIN ANSIEDAD         75
ANSIEDAD LEVE        67
ANSIEDAD MODERADA    45
ANSIEDAD GRAVE       17
Name: count, dtype: int64


In [5]:
X = data[FEATURES]
y = data[TARGET]
groups = None

if "correo" in data.columns and data["correo"].notna().all():
    groups = data["correo"].astype(str).str.strip().str.lower()

if groups is not None and groups.nunique() >= 2:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    train_groups = groups.iloc[train_idx]
    test_groups = groups.iloc[test_idx]
    split_strategy = "GroupShuffleSplit por correo"

    shared_groups = set(train_groups).intersection(set(test_groups))
    print(f"Grupos (correo) compartidos entre train/test: {len(shared_groups)}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    train_groups = None
    split_strategy = "train_test_split estratificado"

print(f"Estrategia de split: {split_strategy}")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

Grupos (correo) compartidos entre train/test: 0
Estrategia de split: GroupShuffleSplit por correo
X_train: (163, 6) | X_test: (41, 6)
y_train: (163,) | y_test: (41,)


In [6]:
baseline_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                solver="lbfgs",
            ),
        ),
    ]
)

if train_groups is not None and train_groups.nunique() >= 5:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(cv.split(X_train, y_train, groups=train_groups))
    cv_strategy = "StratifiedGroupKFold(5)"
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(cv.split(X_train, y_train))
    cv_strategy = "StratifiedKFold(5)"

scoring = {
    "accuracy": "accuracy",
    "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
}
cv_scores = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=cv_splits,
    scoring=scoring,
)

print(f"Estrategia de CV: {cv_strategy}")
print("Cross-validation (baseline)")
for metric in ["test_accuracy", "test_precision_macro", "test_recall_macro", "test_f1_macro"]:
    print(f"{metric}: {cv_scores[metric].mean():.4f} +/- {cv_scores[metric].std():.4f}")

Estrategia de CV: StratifiedGroupKFold(5)
Cross-validation (baseline)
test_accuracy: 0.9206 +/- 0.0489
test_precision_macro: 0.9350 +/- 0.0412
test_recall_macro: 0.9054 +/- 0.0615
test_f1_macro: 0.9137 +/- 0.0525


In [7]:
print("Auditoria anti-fuga de datos")

overlap_idx = set(X_train.index).intersection(set(X_test.index))
print(f"1) Filas compartidas entre train/test: {len(overlap_idx)}")

dup_full = data.duplicated(subset=FEATURES + [TARGET]).sum()
dup_features = data.duplicated(subset=FEATURES).sum()
print(f"2) Duplicados exactos FEATURES+TARGET en dataset: {dup_full}")
print(f"3) Duplicados exactos solo FEATURES en dataset: {dup_features}")

train_patterns = set(map(tuple, X_train[FEATURES].to_numpy()))
test_patterns = set(map(tuple, X_test[FEATURES].to_numpy()))
overlap_patterns = len(train_patterns.intersection(test_patterns))
print(f"4) Patrones de FEATURES repetidos entre train y test: {overlap_patterns}")

if "correo" in data.columns:
    train_emails = set(
        data.loc[X_train.index, "correo"].astype(str).str.strip().str.lower()
    )
    test_emails = set(
        data.loc[X_test.index, "correo"].astype(str).str.strip().str.lower()
    )
    shared_emails = train_emails.intersection(test_emails)
    print(f"5) Correos repetidos entre train y test: {len(shared_emails)}")

    unique_groups = data.loc[X_train.index, "correo"].nunique()
    if unique_groups >= 2:
        from sklearn.model_selection import StratifiedGroupKFold

        n_splits = min(5, unique_groups)
        if n_splits >= 2:
            groups = data.loc[X_train.index, "correo"].astype(str)
            sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
            group_scores = cross_validate(
                baseline_pipeline,
                X_train,
                y_train,
                cv=sgkf.split(X_train, y_train, groups=groups),
                scoring=scoring,
            )
            print(
                "6) Group CV por correo (f1_macro): "
                f"{group_scores['test_f1_macro'].mean():.4f} +/- {group_scores['test_f1_macro'].std():.4f}"
            )
        else:
            print("6) Group CV por correo no ejecutado: grupos insuficientes para split.")
    else:
        print("6) Group CV por correo no ejecutado: no hay grupos suficientes.")
else:
    print("5-6) Columna 'correo' no disponible; se omite validacion por grupo.")

Auditoria anti-fuga de datos
1) Filas compartidas entre train/test: 0
2) Duplicados exactos FEATURES+TARGET en dataset: 70
3) Duplicados exactos solo FEATURES en dataset: 70
4) Patrones de FEATURES repetidos entre train y test: 7
5) Correos repetidos entre train y test: 0
6) Group CV por correo (f1_macro): 0.9092 +/- 0.0247


In [8]:
from collections import Counter
from sklearn.base import clone
from sklearn.metrics import precision_recall_fscore_support

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__class_weight": [None, "balanced"],
}

# 1) Tuning clasico en train para referencia.
grid = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=param_grid,
    cv=cv_splits,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True,
)
grid.fit(X_train, y_train)
best_params_train = grid.best_params_


def _make_grouped_cv(X_data, y_data, groups_data, desired_splits=5, seed=RANDOM_STATE):
    unique_groups_local = groups_data.nunique()
    if unique_groups_local < 3:
        return None, "insufficient_groups"

    n_splits_local = min(desired_splits, unique_groups_local)
    try:
        candidate = StratifiedGroupKFold(
            n_splits=n_splits_local, shuffle=True, random_state=seed
        )
        _ = list(candidate.split(X_data, y_data, groups_data))
        return candidate, f"StratifiedGroupKFold(n_splits={n_splits_local})"
    except ValueError:
        candidate = GroupShuffleSplit(
            n_splits=n_splits_local, test_size=0.2, random_state=seed
        )
        _ = list(candidate.split(X_data, y_data, groups_data))
        return candidate, f"GroupShuffleSplit(n_splits={n_splits_local}, test_size=0.2)"


# 2) Nested grouped CV para estimar generalizacion y robustecer seleccion de hiperparametros.
if "correo" in data.columns:
    groups_full = data.loc[X.index, "correo"].copy()
else:
    groups_full = pd.Series(X.index.astype(str), index=X.index)

outer_cv, outer_strategy = _make_grouped_cv(
    X, y, groups_full, desired_splits=5, seed=RANDOM_STATE
)

nested_fold_metrics = []
nested_best_params = []

if outer_cv is None:
    print("No se pudo ejecutar Nested Grouped CV: grupos insuficientes.")
    nested_cv_summary = {}
else:
    for fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_cv.split(X, y, groups_full), start=1
    ):
        X_outer_train = X.iloc[outer_train_idx]
        y_outer_train = y.iloc[outer_train_idx]
        g_outer_train = groups_full.iloc[outer_train_idx]

        X_outer_test = X.iloc[outer_test_idx]
        y_outer_test = y.iloc[outer_test_idx]

        inner_cv, inner_strategy = _make_grouped_cv(
            X_outer_train,
            y_outer_train,
            g_outer_train,
            desired_splits=4,
            seed=RANDOM_STATE,
        )

        if inner_cv is None:
            continue

        inner_splits = list(inner_cv.split(X_outer_train, y_outer_train, g_outer_train))

        search = GridSearchCV(
            estimator=clone(baseline_pipeline),
            param_grid=param_grid,
            scoring=scoring,
            refit="f1_macro",
            cv=inner_splits,
            n_jobs=-1,
        )
        search.fit(X_outer_train, y_outer_train)

        y_outer_pred = search.predict(X_outer_test)
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y_outer_test, y_outer_pred, average="macro", zero_division=0
        )

        nested_fold_metrics.append(
            {
                "fold": fold,
                "accuracy": accuracy_score(y_outer_test, y_outer_pred),
                "precision_macro": precision_macro,
                "recall_macro": recall_macro,
                "f1_macro": f1_macro,
                "outer_strategy": outer_strategy,
                "inner_strategy": inner_strategy,
            }
        )
        nested_best_params.append(search.best_params_)

    if nested_fold_metrics:
        nested_metrics_df = pd.DataFrame(nested_fold_metrics)
        nested_summary = nested_metrics_df[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].agg(["mean", "std"])

        print("Nested Grouped CV - resumen (mean/std):")
        print(nested_summary)

        nested_cv_summary = {
            "outer_strategy": outer_strategy,
            "n_outer_folds": len(nested_fold_metrics),
            "accuracy_mean": float(nested_summary.loc["mean", "accuracy"]),
            "accuracy_std": float(nested_summary.loc["std", "accuracy"]),
            "precision_macro_mean": float(nested_summary.loc["mean", "precision_macro"]),
            "recall_macro_mean": float(nested_summary.loc["mean", "recall_macro"]),
            "f1_macro_mean": float(nested_summary.loc["mean", "f1_macro"]),
            "f1_macro_std": float(nested_summary.loc["std", "f1_macro"]),
        }
    else:
        nested_cv_summary = {}

# 3) Seleccion final de hiperparametros: moda de Nested CV; fallback al grid clasico.
selected_params = dict(best_params_train)
selection_strategy = "grid_train_best_params"

if nested_best_params:
    params_counter = Counter(tuple(sorted(p.items())) for p in nested_best_params)
    selected_params = dict(params_counter.most_common(1)[0][0])
    selection_strategy = "nested_params_mode"

best_model = clone(baseline_pipeline).set_params(**selected_params)
best_model.fit(X_train, y_train)

print("Hiperparametros seleccionados para exportacion:")
print(selected_params)
print(f"Estrategia de seleccion final: {selection_strategy}")
print(f"Mejor CV train f1_macro (grid clasico): {grid.best_score_:.4f}")

Nested Grouped CV - resumen (mean/std):
      accuracy  precision_macro  recall_macro  f1_macro
mean  0.990122         0.990000      0.966667  0.974737
std   0.013528         0.013693      0.045644  0.034593
Hiperparametros seleccionados para exportacion:
{'model__C': 100.0, 'model__class_weight': None}
Estrategia de seleccion final: nested_params_mode
Mejor CV train f1_macro (grid clasico): 0.9645


In [9]:
y_pred = best_model.predict(X_test)

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
}

print("Metricas en test:")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}")

Metricas en test:
accuracy: 0.9512
precision_macro: 0.9667
recall_macro: 0.9444
f1_macro: 0.9509


In [10]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"actual_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
print("Matriz de confusion:")
cm_df

Matriz de confusion:


,pred_ANSIEDAD GRAVE,pred_ANSIEDAD LEVE,pred_ANSIEDAD MODERADA,pred_SIN ANSIEDAD
actual_ANSIEDAD GRAVE,1,0,0,0
actual_ANSIEDAD LEVE,0,13,0,0
actual_ANSIEDAD MODERADA,0,2,7,0
actual_SIN ANSIEDAD,0,0,0,18


In [11]:
print("Reporte de clasificacion:")
print(classification_report(y_test, y_pred, zero_division=0))

Reporte de clasificacion:
                   precision    recall  f1-score   support

   ANSIEDAD GRAVE       1.00      1.00      1.00         1
    ANSIEDAD LEVE       0.87      1.00      0.93        13
ANSIEDAD MODERADA       1.00      0.78      0.88         9
     SIN ANSIEDAD       1.00      1.00      1.00        18

         accuracy                           0.95        41
        macro avg       0.97      0.94      0.95        41
     weighted avg       0.96      0.95      0.95        41



In [12]:
joblib.dump(best_model, MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")

Modelo guardado en: c:\GitHub\Analitica-Negocios\Semana_1_Proyecto_3\rl_model_ansiedad.joblib


In [13]:
metadata = {
    "model_name": "logistic_regression_ansiedad",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "features": FEATURES,
    "target": TARGET,
    "random_state": RANDOM_STATE,
    "split_strategy": split_strategy,
    "cv_strategy": cv_strategy,
    "train_rows": int(X_train.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "selection_strategy": selection_strategy,
    "best_params": selected_params,
    "train_grid_best_params": best_params_train,
    "cv_best_f1_macro": float(grid.best_score_),
    "nested_cv_summary": nested_cv_summary,
    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
    "artifacts": {
        "model": str(MODEL_PATH),
    },
    "environment": {
        "python": platform.python_version(),
        "sklearn": sklearn_version,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Metadata guardada en: {METADATA_PATH}")

Metadata guardada en: c:\GitHub\Analitica-Negocios\Semana_1_Proyecto_3\rl_model_ansiedad_metadata.json


In [14]:
loaded_model = joblib.load(MODEL_PATH)
example_1 = pd.DataFrame([[2, 3, 2, 2, 2, 2]], columns=FEATURES)
pred_1 = loaded_model.predict(example_1)
print(f"Prediccion ejemplo 1: {pred_1[0]}")

Prediccion ejemplo 1: ANSIEDAD GRAVE


In [15]:
example_2 = pd.DataFrame([[0, 0, 0, 0, 1, 0]], columns=FEATURES)
pred_2 = loaded_model.predict(example_2)
print(f"Prediccion ejemplo 2: {pred_2[0]}")

Prediccion ejemplo 2: SIN ANSIEDAD


## Resumen de seleccion final

Nested Grouped CV ahora esta integrado en la celda de tuning principal para seleccionar hiperparametros finales de exportacion (con fallback al grid clasico).

In [16]:
print("Nested Grouped CV ya se ejecuto dentro de la celda principal de tuning.")
if nested_cv_summary:
    print("Resumen nested:")
    print(nested_cv_summary)
print("Estrategia de seleccion final:", selection_strategy)
print("Hiperparametros exportados:", selected_params)

Nested Grouped CV ya se ejecuto dentro de la celda principal de tuning.
Resumen nested:
{'outer_strategy': 'StratifiedGroupKFold(n_splits=5)', 'n_outer_folds': 5, 'accuracy_mean': 0.9901219512195123, 'accuracy_std': 0.0135277932334877, 'precision_macro_mean': 0.99, 'recall_macro_mean': 0.9666666666666666, 'f1_macro_mean': 0.9747368421052631, 'f1_macro_std': 0.03459300363190521}
Estrategia de seleccion final: nested_params_mode
Hiperparametros exportados: {'model__C': 100.0, 'model__class_weight': None}
